In [ ]:
# setup
from IPython.core.display import display,HTML
display(HTML('<style>.prompt{width: 0px; min-width: 0px; visibility: collapse}</style>'))
display(HTML(open('../rise.css').read()))

# imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style="whitegrid", font_scale=1.5, rc={'figure.figsize':(12, 6)})


<h1>Spanning Trees</h1>

<h3>Last Time</h3>
We started discussing greedy algorithms. They are algorithms that proceed by making locally optimal choices. In special cases, we can argue that the local optimal choices amount to an optimal solution to the problem.

<h3>This Time</h3>
We will see an important example: Spanning Trees. 

<h3>Minimal Spanning Trees</h3>

Let $G$ be a connected graph with non-negative edge weights. A minimal spanning tree is a subset of the edges that forms a spanning tree so that the sum of the weights is minimal.

Since spanning trees are subforest and subforests form a matroid, we get a greedy algorithm to find the minimal spanning tree. This is called Kruskal's algorithm. The idea is to greedily add the lightest edges to the spanning tree as long as the edge does not create a cycle.

We need an additional datastructure to determine if an edge will create a cycle. This is the [union/find](https://en.wikipedia.org/wiki/Disjoint-set_data_structure) data structure. Maybe we should discuss this more later, but it's so fast that we can ignore its cost.

### Kruskal's Algorithm

0. Initialize tree $T \leftarrow \emptyset$  

1. For each $v \in V$, run `make_set(v)`

2. Sort edges in increasing order of weight

3. For each edge $e=(u,v)$ in sorted set:
  - if `find_set(u)` $\ne$ `find_set(v)`:
    - $T \leftarrow T \cup \{(u,v)\}$
    - `union(u,v)`

### Kruskal's Algorithm - work

0. Initialize tree $T \leftarrow \emptyset$
  - $O(1)$

1. For each $v \in V$, run `make_set(v)` 
 - $O(|V|)$

2. Sort edges in increasing order of weight 
  - $O(|E|\log |E|)$

3. For each edge $e=(u,v)$ in sorted set: $O(|E|)$
  - if `find_set(u)` $\ne$ `find_set(v)`: $O(\log |V|)$
    - $T \leftarrow T \cup \{(u,v)\}$
    - `union(u,v)`
  - Together: $O(|E|\log |V|)$

Thus, total work is $O(|E|\log |E|)$.

If the graph is dense, then we have $E \in O(|V|^2)$, and this is equivalent to:

$O(|E|\log |V|^2) = O(2|E| \log |V|) = O(|E| \log |V|)$

<h3>Matroids</h3>

Matroids generalize spanning trees and justify the correctness of Kruskal's algorithm. Here, we describe the matroid structure and prove the correctness of Kruskal's algorithm.

Matroids are significant because they were among the first mathematical objects to be defined by an algorithmic characterization. They provide a theory that relates several Greedy algorithms.

The main reference for this is Greedoids by Lovasz, Korte and Schrader, 1991. This [blog post](https://nor-blog.codeberg.page/posts/2023-01-04-greedoids/) is a short summary. Oxley is a standard reference from a mathematical perspective. A summary of his book is [here](https://www.cs.cornell.edu/courses/cs6820/2022fa/Handouts/oxley-matroids.pdf).

Spanning Trees:
- Let $G$ be a connected graph.
- A subforest subgraph $S\subset G$ is a subgraph $S$ such that $S$ does not have cycles.
- The collection of subforests of $G$ is an example of a matroid.

<h3>Matroid axioms</h3>

A set system $(E,\mathcal{M})$ is a finite set $E$ and a collection of subsets, $\mathcal{M}\subset 2^E$.

A matroid is a set system that satisfies the axioms:
1. (Nontrivial) $\emptyset \in \mathcal{M}$.
2. (Downward closed) If $X\in \mathcal{M}$ and $Y\subset X$, then $Y\in \mathcal{M}$.
3. (Augmentation) If $X,Y\in \mathcal{M}$ and $|X|=|Y|+1$, then there exists $x\in X-Y$ such that $Y\cup\{x\}\in \mathcal{M}$.


For the graphic matroid $(E,\mathcal{M})$ of a graph $G$, the set $E=E(G)$ is the set of edges and $\mathcal{M}\subset{2^E}$ are the subsets of edges that do not have cycles.
Check that subgraphs without cycles both satisfy these axioms.
The main property to check is augmentation.
<details>
<summary>Verifying the augmentation property for graphic matroids.</summary>

Graphic Matroids:
Suppose that $X,Y$ are both subgraphs of a graph $G$ without cycles and that $|E(X)|=|E(Y)|+1$. If $x\in E(X)$ does not have both endpoints in the same connected component as $Y$, then we can add $x$ to $Y$ without creating a cycle. So suppose no $x$ exists. Thus, each edge of $X$ has both endpoints in a connected component of $Y$. Each connected component $C$ of $Y$ can only hold at most $|V(C)|-1$ edges of $X$ and holds exactly $|V(C)|-1$ edges of $Y$. But this means that $|E(X)|\leq |E(Y)|$, which contradicts our assumption.

</details>

<h3>Matroids and greedy algorithms</h3>

Whenever there is a matroid, there is a problem that admits a greedy algorithm.

Let $(E,\mathcal{M})$ be a matroid. Let $w:E\to \mathbb{R}_+$ be a weight function. Find $\max(\{\sum_{v\in M }w(v) :M\in \mathcal{M}\})$.

The greedy algorithm finds a maximal element of $\mathcal{M}$ by greedily choosing the element with the largest $w$.

1. Initialize $M=\emptyset$.
2. While $M$ is not maximal in $\mathcal{M}$ with respect to inclusion:
    1. $v \coloneqq argmax(\{w(v): v\in E-M, M\cup\{v\}\in \mathcal{M}\})$
    2. $M\coloneqq M\cup\{v\}$
3. Return $M$.

By replacing $w(v)$ with $w^\prime(v) = A-w(v)$, for some large number $A$ our greedy algorithm also works on the associated minimization problem.

Here, we prove that the greedy strategy produces optimal results when we are maximizing. The replacement strategy described in the previous paragraph will show that Kruskal's algorithm is correct.

<h3>Proof of matroid greedy strategy</h3>

The proof operates by induction on the size of $M$. Note that this is very similar to our proofs of optimality of greedy algorithms from Day 0. In particular, the strategy is to consider the first choice where the optimal solution and greedy solution differ. Then, we argue that the optimal solution remains optimal if we change that choice so that the optimal solution and greedy solution agree. By repeating this argument, we eventually show that the greedy solution itself is optimal.

1. Let $M$ be the partial solution given by the greedy strategy. Assume that $M$ is optimal for its size: $|M^\prime|= |M| \implies w(M^\prime)\leq w(M)$.
    - This is the inductive hypothesis. (What's the base case?)
2. Suppose that $X\in \mathcal{M}$ satisfies $|X|=|M|+1$ and $X$ is optimal for its size.
3. There exists $x\in X$ such that $M\cup \{x\}\in \mathcal{M}$.
    - Augmentation axiom.
4. $w(X)=\sum_{y\in X} w(y)\geq w(M\cup\{x\})=(\sum_{m\in M}w(m))+w(x)$
    - Since by 2., $X$ is optimal for its size.
5. $\sum_{y \in X-\{x\}} w(y)\geq \sum_{m\in M} w(m)$
    - Removing w(x) from both sums above.
6. $\sum_{y\in X-\{x\}} w(y)\leq\sum_{m\in M}w(m)$
    - optimality of $M$, the downward-closed axiom.
7. $\sum_{y\in X-\{x\}} w(y)=\sum_{m\in M}w(m)$
    - Combine the previous two lines.
8. $\sum_{y\in X} w(y)=\sum_{m\in M\cup \{x\}} w(m)$
    - Add $w(x)$ back to both sides in the sum above.
9. $w(X)= w(M\cup \{x\})$
    - The weight of a set is defined to be the sum of weights of the elements.

This argument shows that $M\cup \{x\}$ is optimal for its size, but the proof is not complete yet. We must establish that $M\cup \{v\}$ is optimal for its size, where $v$ is the element chosen in step 2.1 of the greedy algorithm. This is equivalent to arguing that $w(v)=w(x)$. We get the inequality $w(v)\geq w(x)$ because $v$ was chosen greedily, and $x$ was a possible choice. If the two were not equal, then we would have $w(v)>w(x)$, which would contradict the optimality of $M\cup \{x\}$. This shows that $M\cup \{v\}$ is optimal for its size and re-establishes the inductive hypothesis for the next step.

Review Questions:

1. Translate the proof of optimality of greedy strategies from the context of matroids to the special case of spanning trees.
2. What is the slowest step in Kruskal's algorithm?
3. What Python datastructure should we use to keep track of connected components if we don't want to implement union/find?